In [2]:
import os
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from typing import List, Dict, Any

#import torch
# from torch.utils.data import Dataset
from PIL import Image

from transformers import (
    DetrImageProcessor,
    DetrForObjectDetection,
    TrainingArguments,
    Trainer
)

/opt/anaconda3/envs/advanced_ds/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
! python -c "import platform, sys; print(platform.machine()); print(sys.executable)"

arm64
/opt/anaconda3/envs/advanced_ds/bin/python


In [ ]:


# ============================================================
# 1. CONFIG
# ============================================================

@dataclass
class Config:
    train_image_dir: str = "dataset/train/images"
    train_annot_dir: str = "dataset/train/annotations"
    val_image_dir: str = "dataset/val/images"
    val_annot_dir: str = "dataset/val/annotations"

    pretrained_model_name: str = "facebook/detr-resnet-50"
    output_dir: str = "./detr_voc_finetuned"

    class_names: List[str] = None
    image_size: int = 800

    per_device_train_batch_size: int = 2
    per_device_eval_batch_size: int = 2
    num_train_epochs: int = 15
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    num_workers: int = 2
    save_total_limit: int = 2

    def __post_init__(self):
        # Replace with your actual classes
        if self.class_names is None:
            self.class_names = ["follicle"]


cfg = Config()

# IMPORTANT:
# Pascal VOC class ids must start from 0
id2label = {i: name for i, name in enumerate(cfg.class_names)}
label2id = {name: i for i, name in id2label.items()}

# ============================================================
# 2. BOX HELPERS
# ============================================================

def clip_box_xyxy(box, width, height):
    x_min, y_min, x_max, y_max = box
    x_min = max(0, min(x_min, width - 1))
    y_min = max(0, min(y_min, height - 1))
    x_max = max(0, min(x_max, width - 1))
    y_max = max(0, min(y_max, height - 1))
    return [x_min, y_min, x_max, y_max]

def is_valid_xyxy(box):
    x_min, y_min, x_max, y_max = box
    return (x_max > x_min) and (y_max > y_min)

def xyxy_to_xywh(box):
    x_min, y_min, x_max, y_max = box
    return [x_min, y_min, x_max - x_min, y_max - y_min]

# ============================================================
# 3. PASCAL VOC XML PARSER
# ============================================================

def parse_voc_xml(xml_path: str, label2id: Dict[str, int]) -> Dict[str, Any]:
    tree = ET.parse(xml_path)
    root = tree.getroot()

    filename = root.findtext("filename")
    size = root.find("size")
    width = int(size.findtext("width"))
    height = int(size.findtext("height"))

    objects = []
    for obj in root.findall("object"):
        name = obj.findtext("name")

        # Skip classes not in your label map
        if name not in label2id:
            continue

        bndbox = obj.find("bndbox")
        xmin = float(bndbox.findtext("xmin"))
        ymin = float(bndbox.findtext("ymin"))
        xmax = float(bndbox.findtext("xmax"))
        ymax = float(bndbox.findtext("ymax"))

        difficult = obj.findtext("difficult")
        difficult = int(difficult) if difficult is not None else 0

        objects.append({
            "name": name,
            "class_id": label2id[name],
            "bbox_xyxy": [xmin, ymin, xmax, ymax],
            "difficult": difficult
        })

    return {
        "filename": filename,
        "width": width,
        "height": height,
        "objects": objects
    }

# ============================================================
# 4. DATASET
# ============================================================

class PascalVOCDataset(Dataset):
    def __init__(self, image_dir, annot_dir, processor, label2id):
        self.image_dir = image_dir
        self.annot_dir = annot_dir
        self.processor = processor
        self.label2id = label2id

        self.xml_files = sorted([
            f for f in os.listdir(annot_dir)
            if f.endswith(".xml")
        ])

    def __len__(self):
        return len(self.xml_files)

    def __getitem__(self, idx):
        xml_file = self.xml_files[idx]
        xml_path = os.path.join(self.annot_dir, xml_file)

        parsed = parse_voc_xml(xml_path, self.label2id)

        # Prefer XML filename field, but fallback to xml stem + common extensions
        image_filename = parsed["filename"]
        image_path = os.path.join(self.image_dir, image_filename)

        if not os.path.exists(image_path):
            stem = os.path.splitext(xml_file)[0]
            found = False
            for ext in [".jpg", ".jpeg", ".png", ".bmp"]:
                alt_path = os.path.join(self.image_dir, stem + ext)
                if os.path.exists(alt_path):
                    image_path = alt_path
                    found = True
                    break
            if not found:
                raise FileNotFoundError(f"Image not found for XML: {xml_path}")

        image = Image.open(image_path).convert("RGB")
        width, height = image.size

        annotations = []
        ann_id = 0

        for obj in parsed["objects"]:
            box_xyxy = clip_box_xyxy(obj["bbox_xyxy"], width, height)

            if not is_valid_xyxy(box_xyxy):
                continue

            x, y, w, h = xyxy_to_xywh(box_xyxy)

            annotations.append({
                "id": ann_id,
                "image_id": idx,
                "category_id": int(obj["class_id"]),
                "bbox": [x, y, w, h],   # COCO-style absolute box
                "area": float(w * h),
                "iscrowd": 0
            })
            ann_id += 1

        target = {
            "image_id": idx,
            "annotations": annotations
        }

        encoding = self.processor(
            images=image,
            annotations=target,
            return_tensors="pt"
        )

        pixel_values = encoding["pixel_values"].squeeze(0)
        labels = encoding["labels"][0]

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

# ============================================================
# 5. COLLATE FUNCTION
# ============================================================

processor = DetrImageProcessor.from_pretrained(
    cfg.pretrained_model_name,
    size={"shortest_edge": cfg.image_size, "longest_edge": 1333}
)

def collate_fn(batch):
    pixel_values = [item["pixel_values"] for item in batch]
    labels = [item["labels"] for item in batch]

    encoding = processor.pad(pixel_values, return_tensors="pt")

    return {
        "pixel_values": encoding["pixel_values"],
        "pixel_mask": encoding["pixel_mask"],
        "labels": labels
    }

# ============================================================
# 6. DATASETS
# ============================================================

train_dataset = PascalVOCDataset(
    image_dir=cfg.train_image_dir,
    annot_dir=cfg.train_annot_dir,
    processor=processor,
    label2id=label2id
)

val_dataset = PascalVOCDataset(
    image_dir=cfg.val_image_dir,
    annot_dir=cfg.val_annot_dir,
    processor=processor,
    label2id=label2id
)

# ============================================================
# 7. PRETRAINED MODEL + NEW HEAD
# ============================================================

model = DetrForObjectDetection.from_pretrained(
    cfg.pretrained_model_name,
    num_labels=len(cfg.class_names),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# ============================================================
# 8. TRAINING ARGS
# ============================================================

training_args = TrainingArguments(
    output_dir=cfg.output_dir,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    per_device_eval_batch_size=cfg.per_device_eval_batch_size,
    num_train_epochs=cfg.num_train_epochs,
    learning_rate=cfg.learning_rate,
    weight_decay=cfg.weight_decay,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,
    save_total_limit=cfg.save_total_limit,
    remove_unused_columns=False,
    dataloader_num_workers=cfg.num_workers,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=False
)

# ============================================================
# 9. TRAINER
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=processor,
    data_collator=collate_fn
)

# ============================================================
# 10. TRAIN
# ============================================================

trainer.train()

# ============================================================
# 11. VALIDATION
# ============================================================

eval_results = trainer.evaluate()
print("Validation results:", eval_results)

# ============================================================
# 12. SAVE
# ============================================================

trainer.save_model(cfg.output_dir)
processor.save_pretrained(cfg.output_dir)

# ============================================================
# 13. INFERENCE
# ============================================================

def predict_image(image_path, model, processor, id2label, threshold=0.5, device=None):
    if device is None:
        if torch.cuda.is_available():
            device = "cuda"
        elif torch.backends.mps.is_available():
            device = "mps"
        else:
            device = "cpu"

    model.to(device)
    model.eval()

    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    target_sizes = torch.tensor([image.size[::-1]], device=device)  # (H, W)

    results = processor.post_process_object_detection(
        outputs=outputs,
        threshold=threshold,
        target_sizes=target_sizes
    )[0]

    predictions = []
    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        predictions.append({
            "score": float(score.item()),
            "label_id": int(label.item()),
            "label_name": id2label[int(label.item())],
            "box_xyxy": [round(v, 2) for v in box.tolist()]
        })

    return predictions

# Example:
# preds = predict_image("dataset/val/images/example.jpg", model, processor, id2label, threshold=0.4)
# print(preds)